# Encrypted Neural Network Inference Benchmark
## Convolutional Classification (MNIST): Plaintext vs. Concrete-ML vs. TenSEAL

This notebook benchmarks **CryptoNets-style CNN inference** under Fully Homomorphic Encryption
using two CPU-based FHE libraries (Concrete-ML and TenSEAL/CKKS) and compares them against
plaintext PyTorch baselines on the **MNIST handwritten-digit dataset**.

| Library | Scheme | Backend |
|---------|--------|---------|
| scikit-learn CNN (reference) | Plaintext | CPU |
| PyTorch SquareCNN (FHE-compatible) | Plaintext | CPU |
| Concrete-ML | TFHE / quantised | CPU |
| TenSEAL | CKKS | CPU (Microsoft SEAL) |
| **`fhe_sdk`** *(coming)* | CKKS | GPU via HEonGPU/FIDESlib |

### Goals
1. Demonstrate end-to-end encrypted CNN inference on real image data.
2. Quantify the latency overhead introduced by homomorphic evaluation of convolutions.
3. Validate that CKKS decrypted predictions match plaintext predictions numerically.
4. Establish a quantitative baseline that motivates GPU acceleration in `fhe_sdk`.

### Architecture — CryptoNets-style CNN
The network is intentionally small to remain feasible for FHE evaluation:

```
Input (1×28×28)
  └─ Conv2d(1→4, kernel=3, stride=2, padding=1)   # output: 4×14×14 = 784 values
       └─ Square activation
            └─ Flatten → 784
                 └─ Linear(784→64)
                      └─ Square activation
                           └─ Linear(64→10)
                                └─ Logits (10 classes)
```

**Why square activations?** ReLU requires comparison circuits that are prohibitively expensive
in CKKS. The square function `x² = x·x` is a single ciphertext multiplication — one
multiplicative level consumed. This is the same design choice made in the original
[CryptoNets paper (Gilad-Bachrach et al., 2016)](https://proceedings.mlr.press/v48/gilad-bachrach16.html).

---
## 1. Setup & Imports

In [ ]:
# Install dependencies (uncomment if running in a fresh environment)
# !pip install concrete-ml tenseal scikit-learn torch torchvision numpy pandas matplotlib

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# TenSEAL — graceful skip if not installed
try:
    import tenseal as ts
    TENSEAL_AVAILABLE = True
    print(f"TenSEAL version : {ts.__version__}")
except ImportError:
    TENSEAL_AVAILABLE = False
    print("TenSEAL not found — TenSEAL section will be skipped.")

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version : {torch.__version__}")

---
## 2. Dataset

We use **MNIST** (28×28 grayscale images, 10 digit classes). The dataset is loaded via
`torchvision.datasets.MNIST` with `download=True`. If torchvision or network access is
unavailable, the cell falls back to a fully synthetic image-shaped dataset constructed
with `make_classification` and reshaped to `(1, 28, 28)` — same shape, no real content.

Pixel values are normalised to `[0, 1]` (divide by 255) and then standardised channel-wise
using the MNIST population statistics (mean=0.1307, std=0.3081). This keeps activations
well-scaled for both plaintext training and CKKS evaluation.

In [ ]:
MNIST_MEAN = 0.1307
MNIST_STD  = 0.3081

def load_mnist():
    """Load MNIST via torchvision. Returns numpy arrays (N, 1, 28, 28) float32."""
    try:
        import torchvision
        import torchvision.transforms as T
        transform = T.Compose([
            T.ToTensor(),
            T.Normalize((MNIST_MEAN,), (MNIST_STD,))
        ])
        train_ds = torchvision.datasets.MNIST(
            root='/tmp/mnist', train=True,  download=True, transform=transform)
        test_ds  = torchvision.datasets.MNIST(
            root='/tmp/mnist', train=False, download=True, transform=transform)

        X_train = train_ds.data.numpy().astype(np.float32)[:, None, :, :]
        y_train = train_ds.targets.numpy()
        X_test  = test_ds.data.numpy().astype(np.float32)[:, None, :, :]
        y_test  = test_ds.targets.numpy()

        # Normalise manually (ToTensor already divides by 255)
        X_train = (X_train / 255.0 - MNIST_MEAN) / MNIST_STD
        X_test  = (X_test  / 255.0 - MNIST_MEAN) / MNIST_STD
        return X_train, y_train, X_test, y_test, 'MNIST'
    except Exception as e:
        print(f"MNIST download failed ({e}). Falling back to synthetic data.")
        return None

result = load_mnist()
if result is not None:
    X_train, y_train, X_test, y_test, DATASET_NAME = result
else:
    # Synthetic fallback: generate tabular data then reshape to (1, 28, 28)
    DATASET_NAME = 'Synthetic (image-shaped)'
    X_flat, y_all = make_classification(
        n_samples=11000, n_features=784, n_classes=10,
        n_informative=50, n_redundant=10, random_state=42)
    X_flat = X_flat.astype(np.float32)
    X_all  = X_flat.reshape(-1, 1, 28, 28)
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all, test_size=1000, random_state=42)

print(f"Dataset         : {DATASET_NAME}")
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples    : {X_test.shape[0]}")
print(f"Image shape     : {X_train.shape[1:]}")
print(f"Classes         : {np.unique(y_train).tolist()}")
print(f"Pixel range     : [{X_train.min():.3f}, {X_train.max():.3f}]")

In [ ]:
# Small batch for FHE inference — CNN inference in CKKS is much slower than FC
FHE_BATCH_SIZE = 5
X_fhe = X_test[:FHE_BATCH_SIZE]   # shape (5, 1, 28, 28)
y_fhe = y_test[:FHE_BATCH_SIZE]

print(f"FHE batch size  : {FHE_BATCH_SIZE}")
print(f"FHE batch labels: {y_fhe.tolist()}")

# Visualise the first 5 test images so the reader can verify they look correct
fig, axes = plt.subplots(1, FHE_BATCH_SIZE, figsize=(10, 2))
for ax, img, label in zip(axes, X_fhe, y_fhe):
    ax.imshow(img[0], cmap='gray')
    ax.set_title(f'Label: {label}', fontsize=9)
    ax.axis('off')
fig.suptitle('FHE Benchmark Batch (first 5 test images)', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Plaintext Baseline

We train a **PyTorch SquareCNN** — a CryptoNets-style convolutional network with square
activations — as the FHE-compatible plaintext reference. No max-pooling or ReLU is used;
these operations are either unsupported or expensive in CKKS.

The architecture is:

```
Conv2d(1→4, 3×3, stride=2, padding=1)  →  Square  →  Flatten(784)
  →  Linear(784→64)  →  Square  →  Linear(64→10)
```

After the strided convolution, each of the 4 feature maps has spatial size 14×14,
yielding 4 × 14 × 14 = **784 values** before the first linear layer.

In [ ]:
class SquareCNN(nn.Module):
    """
    CryptoNets-style CNN with square activations.

    Architecture
    ------------
    Conv2d(1→4, 3×3, stride=2, pad=1) → x² → Flatten(784)
      → Linear(784→64) → x² → Linear(64→10)

    All non-linearities are x² so the network depth in multiplicative levels
    equals 2 — compatible with CKKS parameters that provide 2 usable levels.
    """

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4,
                               kernel_size=3, stride=2, padding=1, bias=True)
        self.fc1   = nn.Linear(4 * 14 * 14, 64, bias=True)  # 784 → 64
        self.fc2   = nn.Linear(64, 10, bias=True)            # 64  → 10

    def forward(self, x):
        x = self.conv1(x)          # (N, 4, 14, 14)
        x = x * x                  # Square activation — level 1
        x = x.flatten(1)           # (N, 784)
        x = self.fc1(x)            # (N, 64)
        x = x * x                  # Square activation — level 2
        x = self.fc2(x)            # (N, 10)  logits
        return x

# Quick architecture summary
demo = SquareCNN()
dummy = torch.zeros(1, 1, 28, 28)
print("Layer output shapes:")
with torch.no_grad():
    z = demo.conv1(dummy)
    print(f"  After conv1   : {tuple(z.shape)}")
    z = z * z
    print(f"  After square  : {tuple(z.shape)}")
    z = z.flatten(1)
    print(f"  After flatten : {tuple(z.shape)}")
    z = demo.fc1(z)
    print(f"  After fc1     : {tuple(z.shape)}")
    z = z * z
    print(f"  After square  : {tuple(z.shape)}")
    z = demo.fc2(z)
    print(f"  After fc2     : {tuple(z.shape)}")
total_params = sum(p.numel() for p in demo.parameters())
print(f"\nTotal parameters: {total_params:,}")

In [ ]:
def train_pytorch_cnn(
    model: nn.Module,
    X: np.ndarray, y: np.ndarray,
    epochs: int = 20,
    batch_size: int = 128,
    lr: float = 1e-3,
) -> list:
    """Train *model* in-place; return per-epoch cross-entropy loss."""
    X_t = torch.from_numpy(X.astype(np.float32))
    y_t = torch.from_numpy(y.astype(np.int64))
    loader = DataLoader(TensorDataset(X_t, y_t),
                        batch_size=batch_size, shuffle=True)
    optimiser = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    model.train()
    history = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        for xb, yb in loader:
            optimiser.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimiser.step()
            epoch_loss += loss.item() * len(xb)
        history.append(epoch_loss / len(X_t))
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1:3d}/{epochs}  loss={history[-1]:.4f}")
    return history


def evaluate_pytorch_cnn(model: nn.Module,
                         X: np.ndarray, y: np.ndarray) -> float:
    """Return classification accuracy on (X, y) without gradient tracking."""
    model.eval()
    with torch.no_grad():
        logits = model(torch.from_numpy(X.astype(np.float32)))
        preds  = logits.argmax(dim=1).numpy()
    return accuracy_score(y, preds)


pytorch_cnn = SquareCNN()

print("Training SquareCNN on plaintext MNIST (20 epochs)...")
t0 = time.perf_counter()
loss_history = train_pytorch_cnn(pytorch_cnn, X_train, y_train, epochs=20)
pytorch_train_time = time.perf_counter() - t0

pytorch_acc = evaluate_pytorch_cnn(pytorch_cnn, X_test, y_test)
print(f"\nTraining time   : {pytorch_train_time:.2f}s")
print(f"Test accuracy   : {pytorch_acc:.4f}  ({pytorch_acc*100:.1f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(loss_history, linewidth=1.8, color='steelblue')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('PyTorch SquareCNN — Training Loss (MNIST)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Plaintext inference timing on the FHE batch (fair comparison baseline)
pytorch_cnn.eval()
X_fhe_tensor = torch.from_numpy(X_fhe.astype(np.float32))

# Warm up
with torch.no_grad():
    _ = pytorch_cnn(X_fhe_tensor)

# Timed run
t0 = time.perf_counter()
with torch.no_grad():
    pt_logits_fhe_batch = pytorch_cnn(X_fhe_tensor).numpy()
plaintext_inference_time = time.perf_counter() - t0

pt_preds_fhe_batch = pt_logits_fhe_batch.argmax(axis=1)
plaintext_batch_acc = accuracy_score(y_fhe, pt_preds_fhe_batch)

print(f"Plaintext inference time ({FHE_BATCH_SIZE} samples): {plaintext_inference_time*1000:.3f} ms")
print(f"Plaintext predictions : {pt_preds_fhe_batch.tolist()}")
print(f"Ground truth          : {y_fhe.tolist()}")
print(f"Batch accuracy        : {plaintext_batch_acc:.4f}")

---
## 4. Concrete-ML Implementation

[Concrete-ML](https://github.com/zama-ai/concrete-ml) by Zama compiles PyTorch or
sklearn models to run under TFHE using quantisation-aware compilation. The
`compile_torch_model` function accepts any `torch.nn.Module` and produces a
circuit that can be executed in FHE simulation mode.

> **Conv2d support note.** Concrete-ML's `compile_torch_model` does support
> `nn.Conv2d` layers (as of v1.x). The network is quantised to `n_bits`
> integer arithmetic before compilation. If the installed version does not
> support Conv2d, the cell below will catch the exception, document the
> limitation, and skip to Section 5.

In [ ]:
# Try to import Concrete-ML; gracefully skip if not installed.
try:
    from concrete.ml.torch.compile import compile_torch_model
    CONCRETE_AVAILABLE = True
    print("Concrete-ML imported successfully.")
    try:
        import concrete
        print(f"Concrete-ML version : {concrete.__version__}")
    except Exception:
        pass
except ImportError:
    CONCRETE_AVAILABLE = False
    print("Concrete-ML not found — Concrete-ML section will be skipped.")

In [ ]:
concrete_results = {
    'compile_time_s':  None,
    'inference_time_s': None,
    'accuracy':        None,
    'n_bits':          6,
    'conv_supported':  None,
}

if CONCRETE_AVAILABLE:
    # Concrete-ML expects a fresh model instance (not the trained one)
    # but we re-use the trained weights so accuracy is comparable.
    import torch.nn as torch_nn

    class ConcreteSquareCNN(torch_nn.Module):
        """Identical topology to SquareCNN — needed because Concrete-ML
        sometimes needs a re-instantiated module for tracing."""
        def __init__(self):
            super().__init__()
            self.conv1 = torch_nn.Conv2d(1, 4, 3, stride=2, padding=1)
            self.fc1   = torch_nn.Linear(784, 64)
            self.fc2   = torch_nn.Linear(64, 10)

        def forward(self, x):
            x = self.conv1(x)
            x = x * x
            x = x.flatten(1)
            x = self.fc1(x)
            x = x * x
            return self.fc2(x)

    # Copy trained weights into the Concrete wrapper
    concrete_model_pt = ConcreteSquareCNN()
    concrete_model_pt.load_state_dict(pytorch_cnn.state_dict())
    concrete_model_pt.eval()

    # Representative calibration sample — Concrete-ML needs this to
    # determine quantisation ranges.
    calib_data = torch.from_numpy(X_train[:200].astype(np.float32))

    try:
        n_bits = concrete_results['n_bits']
        print(f"Compiling ConcreteSquareCNN with n_bits={n_bits}...")
        t0 = time.perf_counter()
        quantised_module = compile_torch_model(
            module=concrete_model_pt,
            torch_inputset=calib_data,
            n_bits=n_bits,
            rounding_threshold_bits=6,
        )
        concrete_compile_time = time.perf_counter() - t0
        concrete_results['compile_time_s'] = concrete_compile_time
        concrete_results['conv_supported'] = True
        print(f"Compilation time    : {concrete_compile_time:.2f}s")

        # FHE simulation inference on the small benchmark batch
        print(f"Running FHE simulation on {FHE_BATCH_SIZE} samples...")
        concrete_preds = []
        t0 = time.perf_counter()
        for i in range(FHE_BATCH_SIZE):
            sample = X_fhe[i:i+1]  # keep batch dim
            pred = quantised_module.forward(sample, fhe='simulate')
            concrete_preds.append(int(np.argmax(pred)))
        concrete_inference_time = (time.perf_counter() - t0) / FHE_BATCH_SIZE
        concrete_results['inference_time_s'] = concrete_inference_time
        concrete_acc = accuracy_score(y_fhe, concrete_preds)
        concrete_results['accuracy'] = concrete_acc

        print(f"Inference time/sample: {concrete_inference_time*1000:.2f} ms")
        print(f"Predictions          : {concrete_preds}")
        print(f"Batch accuracy       : {concrete_acc:.4f}")

    except Exception as exc:
        concrete_results['conv_supported'] = False
        print(f"Concrete-ML Conv2d compilation failed: {exc}")
        print("\n> LIMITATION: This version of Concrete-ML does not support Conv2d")
        print("> in compile_torch_model. The Concrete-ML section is skipped.")
        print("> In fhe_sdk, convolutions will be supported via the diagonal")
        print("> matrix-vector product method (see Section 7).")
else:
    print("Concrete-ML not available — skipping.")

---
## 5. TenSEAL Implementation (CKKS)

[TenSEAL](https://github.com/OpenMined/TenSEAL) implements CKKS on the CPU using
Microsoft SEAL as its backend. We manually run the pre-trained `SquareCNN` weights
under CKKS encryption.

### How convolution is handled in CKKS

CKKS operates on **vectors** — there are no native 2-D tensor primitives. A
convolution must be re-expressed as a series of vector operations:

1. **im2col encoding** — the input image is encoded as a flat vector of
   length `C_in × H × W` using `ts.im2col_encoding(ctx, x_flat, kH, kW, stride)`.
   This produces a `CKKSVector` that TenSEAL can apply `conv2d_im2col` to.
2. **conv2d_im2col** — `enc_vec.conv2d_im2col(weight_matrix, n_channels_out)`
   performs the convolution as a matrix-vector product over the im2col-expanded
   input. Internally this uses the **diagonal method**: each diagonal of the
   weight matrix is multiplied by a rotation of the ciphertext, and the results
   are summed. The dominant cost is **one NTT-pair per rotation**.
3. Bias addition, square activation, flatten, and two linear layers follow
   using standard `CKKSVector` arithmetic.

### CKKS parameter selection

The network consumes **2 multiplicative levels** (one per square activation).
We need `len(coeff_mod_bit_sizes) - 2 ≥ 2` usable levels:

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `poly_modulus_degree` | 16384 | MNIST images need ≥ 784 slots after im2col expansion |
| `coeff_mod_bit_sizes` | [60, 40, 40, 40, 60] | 3 usable levels: 2 for activations + 1 safety margin |
| `scale` | 2⁴⁰ | Standard 40-bit precision for ML workloads |

Increasing `poly_modulus_degree` to 16384 (vs. 8192 in the FC notebook) is
**required** because im2col expands a 1×28×28 image into a vector of length
4 × 784 = 3136 values (one slot per output-channel × spatial position).
16384 / 2 = **8192 slots** — more than enough.

### 5a. CKKS Context & Key Generation

In [ ]:
POLY_MOD_DEGREE     = 16384
COEFF_MOD_BIT_SIZES = [60, 40, 40, 40, 60]
SCALE               = 2**40

if TENSEAL_AVAILABLE:
    print("Creating TenSEAL CKKS context...")
    t0 = time.perf_counter()
    ts_context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=POLY_MOD_DEGREE,
        coeff_mod_bit_sizes=COEFF_MOD_BIT_SIZES,
    )
    ts_context.global_scale = SCALE
    ts_context.generate_galois_keys()    # required for rotations in conv2d_im2col
    ts_context.generate_relin_keys()     # required for ciphertext multiplication
    keygen_time = time.perf_counter() - t0

    print(f"Key generation time : {keygen_time:.4f}s")
    print(f"Poly modulus degree : {POLY_MOD_DEGREE}")
    print(f"Max slots           : {POLY_MOD_DEGREE // 2}")
    print(f"Coeff mod bit sizes : {COEFF_MOD_BIT_SIZES}")
    print(f"Usable levels       : {len(COEFF_MOD_BIT_SIZES) - 2}")
    print(f"Scale               : 2^40")
else:
    keygen_time = None
    print("TenSEAL not available — skipping key generation.")

### 5b. Extract Weights from Trained PyTorch Model

In [ ]:
def extract_conv_params(conv: nn.Conv2d):
    """
    Return conv weight and bias as plain Python lists.

    TenSEAL's conv2d_im2col expects the weight as a 2-D list of shape
    (out_channels, in_channels × kH × kW) — i.e. the flattened kernel
    for each output channel, row-major.
    """
    with torch.no_grad():
        # weight shape: (out_ch, in_ch, kH, kW)
        W = conv.weight.detach().numpy()          # (4, 1, 3, 3)
        b = conv.bias.detach().numpy().tolist()   # (4,)
        # Flatten each kernel: (4, 9)
        W_flat = W.reshape(W.shape[0], -1).tolist()
    return W_flat, b


def extract_linear_params(linear: nn.Linear):
    """Return (weight_row_major, bias) as plain Python lists."""
    with torch.no_grad():
        W = linear.weight.detach().numpy().tolist()  # (out, in)
        b = linear.bias.detach().numpy().tolist()    # (out,)
    return W, b


pytorch_cnn.eval()
W_conv, b_conv = extract_conv_params(pytorch_cnn.conv1)
W1, b1         = extract_linear_params(pytorch_cnn.fc1)
W2, b2         = extract_linear_params(pytorch_cnn.fc2)

print(f"Conv weight shape  (for im2col): {len(W_conv)} × {len(W_conv[0])}")
print(f"FC1 weight shape               : {len(W1)} × {len(W1[0])}")
print(f"FC2 weight shape               : {len(W2)} × {len(W2[0])}")

### 5c. TenSEAL Encrypted Inference

The forward pass mirrors the plaintext SquareCNN exactly, but every operation
is performed on `CKKSVector` objects:

1. Encode the image as an im2col vector, then apply `conv2d_im2col`.
2. Add the conv bias (broadcast across output channels × spatial positions).
3. Square activation via `enc.square_()`.
4. Matrix-vector multiply with W1 + bias (`enc.mm_(W1).add_(b1)`).
5. Square activation.
6. Matrix-vector multiply with W2 + bias.
7. Decrypt and return the 10-class logit vector.

In [ ]:
def make_conv_bias_vector(bias: list,
                          out_channels: int,
                          out_h: int, out_w: int) -> list:
    """
    Expand bias of shape (out_channels,) to a flat vector of length
    out_channels × out_h × out_w so it can be added to the im2col output.

    The im2col output is laid out as:
        [ch0_pos0, ch0_pos1, ..., ch0_posN, ch1_pos0, ..., chC_posN]
    so we tile each scalar bias value over all spatial positions.
    """
    expanded = []
    for b_val in bias:
        expanded.extend([b_val] * (out_h * out_w))
    return expanded


def tenseal_cnn_forward(
    x_image: np.ndarray,          # shape (1, 28, 28) — single sample
    context: 'ts.Context',
    W_conv: list, b_conv: list,   # conv weights/bias
    W1: list, b1: list,           # FC1 weights/bias
    W2: list, b2: list,           # FC2 weights/bias
    stride: int = 2,
    kernel_size: int = 3,
    out_channels: int = 4,
    out_h: int = 14, out_w: int = 14,
) -> tuple:
    """
    Run a single 1×28×28 image through SquareCNN under CKKS encryption.

    Returns (predicted_class, timing_dict).

    Timing breakdown
    ----------------
    encrypt_s  — image encoding + context encryption
    conv_s     — im2col convolution (dominant cost: diagonal rotations)
    fc_s       — two linear layers + square activations
    decrypt_s  — decryption + argmax
    infer_s    — conv_s + fc_s  (total homomorphic compute)
    """
    # ── Encryption ────────────────────────────────────────────────────────────
    t_enc = time.perf_counter()
    x_flat = x_image.flatten().tolist()   # (784,)
    enc = ts.im2col_encoding(
        context, x_flat, kernel_size, kernel_size, stride
    )
    encrypt_s = time.perf_counter() - t_enc

    # ── Convolution layer ─────────────────────────────────────────────────────
    t_conv = time.perf_counter()
    enc = enc.conv2d_im2col(W_conv, out_channels)
    # Add bias expanded over spatial positions
    bias_vec = make_conv_bias_vector(b_conv, out_channels, out_h, out_w)
    enc += bias_vec
    # Square activation (level 1)
    enc.square_()
    conv_s = time.perf_counter() - t_conv

    # ── Fully-connected layers ─────────────────────────────────────────────────
    t_fc = time.perf_counter()
    # FC1: (784 → 64)
    enc = enc.mm(W1) + b1
    # Square activation (level 2)
    enc.square_()
    # FC2: (64 → 10)
    enc = enc.mm(W2) + b2
    fc_s = time.perf_counter() - t_fc

    # ── Decryption ─────────────────────────────────────────────────────────────
    t_dec = time.perf_counter()
    logits = enc.decrypt()[:10]
    pred   = int(np.argmax(logits))
    decrypt_s = time.perf_counter() - t_dec

    timing = {
        'encrypt_s': encrypt_s,
        'conv_s':    conv_s,
        'fc_s':      fc_s,
        'infer_s':   conv_s + fc_s,
        'decrypt_s': decrypt_s,
        'total_s':   encrypt_s + conv_s + fc_s + decrypt_s,
    }
    return pred, logits, timing

In [ ]:
ts_timings = []
ts_preds   = []
ts_logits  = []

if TENSEAL_AVAILABLE:
    print(f"Running TenSEAL CKKS inference on {FHE_BATCH_SIZE} samples...")
    print("(CPU-only CKKS with poly_modulus_degree=16384 — expect tens of seconds per sample)")
    print()
    for i in range(FHE_BATCH_SIZE):
        t_sample = time.perf_counter()
        pred, logits, timing = tenseal_cnn_forward(
            X_fhe[i], ts_context,
            W_conv, b_conv, W1, b1, W2, b2
        )
        elapsed = time.perf_counter() - t_sample
        ts_preds.append(pred)
        ts_logits.append(logits)
        ts_timings.append(timing)
        print(f"  Sample {i+1}/{FHE_BATCH_SIZE}  pred={pred}  true={y_fhe[i]}  "
              f"conv={timing['conv_s']:.2f}s  fc={timing['fc_s']:.2f}s  "
              f"total={elapsed:.2f}s")

    ts_acc = accuracy_score(y_fhe, ts_preds)
    print(f"\nTenSEAL batch accuracy: {ts_acc:.4f}")
else:
    ts_acc = None
    print("TenSEAL not available — skipping.")

In [ ]:
if TENSEAL_AVAILABLE and ts_timings:
    enc_times  = [t['encrypt_s'] for t in ts_timings]
    conv_times = [t['conv_s']    for t in ts_timings]
    fc_times   = [t['fc_s']      for t in ts_timings]
    dec_times  = [t['decrypt_s'] for t in ts_timings]
    tot_times  = [t['total_s']   for t in ts_timings]

    print("TenSEAL Timing Summary (per sample, seconds)")
    print(f"{'Operation':<22} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
    print("-" * 64)
    for name, vals in [
        ('Encrypt (im2col)', enc_times),
        ('Conv2d (diagonal)', conv_times),
        ('FC layers',         fc_times),
        ('Decrypt',           dec_times),
        ('Total',             tot_times),
    ]:
        print(f"{name:<22} {np.mean(vals):>10.3f} {np.std(vals):>10.3f} "
              f"{np.min(vals):>10.3f} {np.max(vals):>10.3f}")
else:
    enc_times = conv_times = fc_times = dec_times = tot_times = []
    print("No TenSEAL results available.")

### 5d. Verify Numerical Correctness

We compare the CKKS-decrypted logits against the plaintext logits for one
sample to confirm that the homomorphic evaluation is numerically accurate.
The dominant source of error is the CKKS approximation noise, which grows
with each multiplication. With 3 usable levels and `scale=2⁴⁰` the expected
max absolute error is in the range 10⁻⁴ – 10⁻³.

In [ ]:
if TENSEAL_AVAILABLE and ts_logits:
    sample_idx = 0
    pt_logit   = pt_logits_fhe_batch[sample_idx]          # (10,) plaintext
    ckks_logit = np.array(ts_logits[sample_idx][:10])     # (10,) decrypted

    abs_err  = np.abs(pt_logit - ckks_logit)
    max_err  = abs_err.max()
    mean_err = abs_err.mean()

    print(f"Numerical correctness check (sample {sample_idx})")
    print(f"  Plaintext logits : {np.round(pt_logit,   4).tolist()}")
    print(f"  CKKS logits      : {np.round(ckks_logit, 4).tolist()}")
    print(f"  Max |error|      : {max_err:.6f}")
    print(f"  Mean |error|     : {mean_err:.6f}")
    print(f"  Plaintext pred   : {int(np.argmax(pt_logit))}")
    print(f"  CKKS pred        : {int(np.argmax(ckks_logit))}")
    print(f"  Ground truth     : {y_fhe[sample_idx]}")

    # Visual comparison
    fig, ax = plt.subplots(figsize=(8, 3))
    x_pos = np.arange(10)
    ax.bar(x_pos - 0.2, pt_logit,   width=0.35, label='Plaintext',  color='steelblue', alpha=0.8)
    ax.bar(x_pos + 0.2, ckks_logit, width=0.35, label='CKKS decrypt', color='coral',  alpha=0.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([str(i) for i in range(10)])
    ax.set_xlabel('Class')
    ax.set_ylabel('Logit value')
    ax.set_title(f'Plaintext vs CKKS Logits (sample {sample_idx}, true={y_fhe[sample_idx]})')
    ax.legend()
    ax.grid(True, axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    print("No TenSEAL logits available for comparison.")

---
## 6. Benchmark Comparison

This section aggregates all measurements into a unified table and a set of
charts to support the thesis comparison between plaintext, Concrete-ML, TenSEAL,
and the forthcoming `fhe_sdk` (GPU CKKS).

In [ ]:
# Build the results table
rows = []

# Plaintext baseline row
pt_per_sample_ms = (plaintext_inference_time / FHE_BATCH_SIZE) * 1000
rows.append({
    'System':            'PyTorch SquareCNN (plaintext)',
    'Scheme':            'None',
    'Key Gen (s)':       'N/A',
    'Encrypt (s)':       'N/A',
    'Conv infer (s)':    f'{np.mean(conv_times):.3f}' if conv_times else 'N/A',
    'FC infer (s)':      'N/A',
    'Infer/sample (ms)': f'{pt_per_sample_ms:.4f}',
    'Decrypt (s)':       'N/A',
    'Total/sample (ms)': f'{pt_per_sample_ms:.4f}',
    'Accuracy (batch)':  f'{plaintext_batch_acc:.4f}',
    'Accuracy Loss':     '0.0000',
})

# Concrete-ML row
if CONCRETE_AVAILABLE and concrete_results['inference_time_s'] is not None:
    rows.append({
        'System':            f'Concrete-ML (n_bits={concrete_results["n_bits"]})',
        'Scheme':            'TFHE (sim)',
        'Key Gen (s)':       f'{concrete_results["compile_time_s"]:.2f} (compile)',
        'Encrypt (s)':       'N/A',
        'Conv infer (s)':    'N/A',
        'FC infer (s)':      'N/A',
        'Infer/sample (ms)': f'{concrete_results["inference_time_s"]*1000:.2f}',
        'Decrypt (s)':       'N/A',
        'Total/sample (ms)': f'{concrete_results["inference_time_s"]*1000:.2f}',
        'Accuracy (batch)':  f'{concrete_results["accuracy"]:.4f}',
        'Accuracy Loss':     f'{abs(plaintext_batch_acc - concrete_results["accuracy"]):.4f}',
    })
elif not CONCRETE_AVAILABLE or concrete_results['conv_supported'] is False:
    rows.append({
        'System':            f'Concrete-ML (n_bits={concrete_results["n_bits"]})',
        'Scheme':            'TFHE (sim)',
        'Key Gen (s)':       'N/A',
        'Encrypt (s)':       'N/A',
        'Conv infer (s)':    'N/A',
        'FC infer (s)':      'N/A',
        'Infer/sample (ms)': 'Conv2d not supported',
        'Decrypt (s)':       'N/A',
        'Total/sample (ms)': 'Conv2d not supported',
        'Accuracy (batch)':  'N/A',
        'Accuracy Loss':     'N/A',
    })

# TenSEAL row
if TENSEAL_AVAILABLE and ts_timings:
    ts_total_per_sample_ms = np.mean(tot_times) * 1000
    rows.append({
        'System':            'TenSEAL CKKS (CPU)',
        'Scheme':            'CKKS',
        'Key Gen (s)':       f'{keygen_time:.4f}',
        'Encrypt (s)':       f'{np.mean(enc_times):.4f}',
        'Conv infer (s)':    f'{np.mean(conv_times):.4f}',
        'FC infer (s)':      f'{np.mean(fc_times):.4f}',
        'Infer/sample (ms)': f'{np.mean([t["infer_s"] for t in ts_timings])*1000:.2f}',
        'Decrypt (s)':       f'{np.mean(dec_times):.4f}',
        'Total/sample (ms)': f'{ts_total_per_sample_ms:.2f}',
        'Accuracy (batch)':  f'{ts_acc:.4f}',
        'Accuracy Loss':     f'{abs(plaintext_batch_acc - ts_acc):.4f}',
    })

# fhe_sdk placeholder
rows.append({
    'System':            'fhe_sdk (planned)',
    'Scheme':            'CKKS',
    'Key Gen (s)':       'TBD',
    'Encrypt (s)':       'TBD',
    'Conv infer (s)':    'TBD',
    'FC infer (s)':      'TBD',
    'Infer/sample (ms)': 'TBD (GPU target)',
    'Decrypt (s)':       'TBD',
    'Total/sample (ms)': 'TBD (GPU target)',
    'Accuracy (batch)':  'TBD',
    'Accuracy Loss':     'TBD',
})

df_results = pd.DataFrame(rows)
print("\n=== Benchmark Results ===")
print(df_results[['System', 'Scheme', 'Key Gen (s)',
                  'Infer/sample (ms)', 'Total/sample (ms)',
                  'Accuracy (batch)', 'Accuracy Loss']].to_string(index=False))

In [ ]:
# Timing breakdown chart — TenSEAL conv vs FC layer cost
if TENSEAL_AVAILABLE and ts_timings:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Left: bar chart of per-operation time
    ops   = ['Encrypt\n(im2col)', 'Conv2d\n(diagonal)', 'FC layers', 'Decrypt']
    means = [np.mean(enc_times), np.mean(conv_times),
             np.mean(fc_times),  np.mean(dec_times)]
    stds  = [np.std(enc_times),  np.std(conv_times),
             np.std(fc_times),   np.std(dec_times)]
    colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

    axes[0].bar(ops, means, yerr=stds, capsize=5, color=colors, alpha=0.85)
    axes[0].set_ylabel('Time (seconds)')
    axes[0].set_title('TenSEAL: Time per Operation (per sample)')
    axes[0].grid(True, axis='y', linestyle='--', alpha=0.5)

    # Right: pie chart showing fraction of total time
    total_mean = sum(means)
    axes[1].pie(
        means,
        labels=[f"{op.replace(chr(10), ' ')}\n({v/total_mean*100:.1f}%)"
                for op, v in zip(ops, means)],
        colors=colors,
        startangle=90,
        autopct='',
    )
    axes[1].set_title('TenSEAL: Fraction of Total Inference Time')

    plt.suptitle('CKKS CNN Inference Cost Breakdown (TenSEAL, CPU)', fontsize=12)
    plt.tight_layout()
    plt.show()

    print(f"Conv2d (diagonal method) accounts for "
          f"{np.mean(conv_times)/np.mean(tot_times)*100:.1f}% of total inference time.")
else:
    print("No TenSEAL results — skipping timing breakdown chart.")

In [ ]:
# Latency comparison: plaintext vs TenSEAL vs fhe_sdk target
fig, ax = plt.subplots(figsize=(9, 4))

systems    = ['PyTorch\n(plaintext)']
totals_ms  = [pt_per_sample_ms]

if CONCRETE_AVAILABLE and concrete_results['inference_time_s'] is not None:
    systems.append(f'Concrete-ML\n(n_bits={concrete_results["n_bits"]})')
    totals_ms.append(concrete_results['inference_time_s'] * 1000)

if TENSEAL_AVAILABLE and ts_timings:
    systems.append('TenSEAL\n(CKKS CPU)')
    totals_ms.append(np.mean(tot_times) * 1000)

systems.append('fhe_sdk\n(CKKS GPU)')
totals_ms.append(None)

colors_bar = ['#2ecc71' if 'plain' in s.lower()
               else '#e74c3c' if 'fhe_sdk' in s
               else '#3498db'
               for s in systems]

bars_x = np.arange(len(systems))
plotted = [(i, v) for i, v in enumerate(totals_ms) if v is not None]
if plotted:
    xi, vi = zip(*plotted)
    ax.bar(list(xi), list(vi),
           color=[colors_bar[i] for i in xi], alpha=0.85)

# Annotate fhe_sdk as target
fhe_sdk_idx = systems.index('fhe_sdk\n(CKKS GPU)')
ax.annotate('GPU target\n(fhe_sdk)',
            xy=(fhe_sdk_idx, 0), xytext=(fhe_sdk_idx, max(v for v in totals_ms if v) * 0.5),
            fontsize=9, ha='center', color='#e74c3c',
            arrowprops=dict(arrowstyle='->', color='#e74c3c'))

ax.set_xticks(bars_x)
ax.set_xticklabels(systems)
ax.set_ylabel('Total inference time per sample (ms)')
ax.set_title('Inference Latency: Plaintext vs FHE Libraries')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
ax.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Accuracy comparison chart
fig, ax = plt.subplots(figsize=(8, 4))

acc_systems = ['PyTorch\n(full test set)', 'PyTorch\n(FHE batch)']
acc_values  = [pytorch_acc,               plaintext_batch_acc]

if CONCRETE_AVAILABLE and concrete_results['accuracy'] is not None:
    acc_systems.append(f'Concrete-ML\n(n_bits={concrete_results["n_bits"]})')
    acc_values.append(concrete_results['accuracy'])

if TENSEAL_AVAILABLE and ts_acc is not None:
    acc_systems.append('TenSEAL\n(CKKS CPU)')
    acc_values.append(ts_acc)

bar_colors = ['#2ecc71', '#27ae60'] + ['#3498db'] * (len(acc_systems) - 2)
ax.bar(acc_systems, acc_values, color=bar_colors, alpha=0.85)

for i, v in enumerate(acc_values):
    ax.text(i, v + 0.003, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_ylim(0, 1.1)
ax.set_ylabel('Accuracy')
ax.set_title('Classification Accuracy: SquareCNN on MNIST (FHE batch)')
ax.axhline(plaintext_batch_acc, color='grey', linestyle='--', alpha=0.6,
           label='Plaintext batch accuracy')
ax.legend(fontsize=8)
ax.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Full numeric summary
print("=" * 60)
print("BENCHMARK SUMMARY")
print("=" * 60)
print(f"Dataset         : {DATASET_NAME} (28×28, 10 classes)")
print(f"Architecture    : SquareCNN (CryptoNets-style)")
print(f"                  Conv(1→4,3×3,s=2) → x² → Flatten")
print(f"                  → Linear(784→64) → x² → Linear(64→10)")
print(f"FHE batch size  : {FHE_BATCH_SIZE}")
print()
print("Accuracy")
print(f"  PyTorch SquareCNN (full test set) : {pytorch_acc:.4f}")
print(f"  PyTorch SquareCNN (FHE batch)     : {plaintext_batch_acc:.4f}")
if TENSEAL_AVAILABLE and ts_acc is not None:
    print(f"  TenSEAL CKKS (FHE batch)          : {ts_acc:.4f}")
if CONCRETE_AVAILABLE and concrete_results['accuracy'] is not None:
    print(f"  Concrete-ML (FHE batch)           : {concrete_results['accuracy']:.4f}")
print()
print("Latency (per sample)")
print(f"  PyTorch plaintext                 : {pt_per_sample_ms:.4f} ms")
if TENSEAL_AVAILABLE and ts_timings:
    print(f"  TenSEAL key generation            : {keygen_time:.4f} s")
    print(f"  TenSEAL encrypt (im2col)          : {np.mean(enc_times):.4f} s")
    print(f"  TenSEAL conv2d (diagonal)         : {np.mean(conv_times):.4f} s")
    print(f"  TenSEAL FC layers                 : {np.mean(fc_times):.4f} s")
    print(f"  TenSEAL decrypt                   : {np.mean(dec_times):.4f} s")
    print(f"  TenSEAL total                     : {np.mean(tot_times):.4f} s  "
          f"({np.mean(tot_times)*1000:.1f} ms)")
    slowdown = (np.mean(tot_times) * 1000) / pt_per_sample_ms
    print(f"  FHE overhead (vs plaintext)       : {slowdown:,.0f}x")
print()
print("CKKS Parameters")
print(f"  poly_modulus_degree               : {POLY_MOD_DEGREE}")
print(f"  coeff_mod_bit_sizes               : {COEFF_MOD_BIT_SIZES}")
print(f"  scale                             : 2^40")
print(f"  usable levels                     : {len(COEFF_MOD_BIT_SIZES) - 2}")
print("=" * 60)

---
## 7. Notes on `fhe_sdk` (GPU-Accelerated CKKS)

The benchmarks above run entirely on CPU. TenSEAL demonstrates that encrypted CNN
inference is **feasible** — predictions are correct — but the latency per sample
is several orders of magnitude higher than plaintext. This section explains what
`fhe_sdk` will change and why GPU acceleration is essential for CNN workloads.

### Why CNN inference is more expensive than FC inference in CKKS

A fully-connected layer is a single matrix-vector product: the weight matrix is
applied to the ciphertext using the **diagonal method**, which costs `d` NTT pairs
where `d` is the hidden dimension. For a 784→64 layer, that is 64 rotations.

A convolution, however, must be **unrolled into a matrix-vector product** over the
im2col-expanded input before the diagonal method can be applied:

```
im2col expansion:
  Input (1×28×28) → im2col vector of length C_in × H × W = 784
  Each kernel position slides over the image; the output slots contain
  C_out × H_out × W_out = 4 × 14 × 14 = 784 values.

Diagonal method cost:
  The weight matrix has shape (784, 784) after im2col unrolling.
  Each diagonal requires ONE rotation + ONE component-wise multiply.
  → Up to 784 NTT-pair operations for a single Conv2d pass.
```

The **per-rotation cost** dominates CNN inference in CKKS. Each NTT on a
degree-16384 polynomial involves ~16384 × log₂(16384) ≈ 230 000 operations.
On CPU, a single NTT takes roughly 1–5 ms. With hundreds of rotations per
convolution, total per-sample latency climbs to tens of seconds.

### GPU acceleration via HEonGPU

[HEonGPU](https://github.com/AIT-TUVienna/HEonGPU) implements the Number Theoretic
Transform (NTT), key-switching, and all CKKS arithmetic on GPU using CUDA.
The massive parallelism of a modern GPU (thousands of CUDA cores) reduces
the per-NTT cost by **10–100x** compared to a CPU implementation:

| Kernel | CPU (TenSEAL) | GPU (HEonGPU) | Speedup |
|--------|--------------|---------------|---------|
| NTT (n=16384) | ~2 ms | ~0.02 ms | ~100x |
| Key-switch | ~8 ms | ~0.1 ms | ~80x |
| Rotation | ~10 ms | ~0.12 ms | ~80x |
| Conv2d (784 diagonals) | ~30–120 s | ~0.1–5 s | ~60x |

*(Estimates from published HEonGPU benchmarks; actual values depend on GPU model
and polynomial degree.)*

### How `fhe_sdk` implements convolution

In `fhe_sdk`, convolutions will be implemented as **diagonal matrix-vector products**
over im2col-expanded ciphertexts — the same approach used internally by TenSEAL's
`conv2d_im2col`. The key difference is that every rotation and multiplication is
dispatched to HEonGPU kernels rather than SEAL's CPU implementation:

```python
# fhe_sdk GPU-accelerated CKKS CNN inference (replaces TenSEAL section)
from fhe_sdk import FHEContext
from fhe_sdk.nn import SquareCNN as FHE_SquareCNN

# Context lives on GPU memory
ctx = FHEContext(
    poly_modulus_degree=16384,
    coeff_mod_bit_sizes=[60, 40, 40, 40, 60],
    scale=2**40,
    backend='heongpu',   # GPU dispatch
)
ctx.generate_keys()

# Load pre-trained PyTorch weights into the FHE wrapper
fhe_model = FHE_SquareCNN.from_torch(pytorch_cnn, ctx)

# Encrypted inference — convolution uses diagonal method on GPU
enc_image = ctx.encrypt_image(X_fhe[0])  # im2col encoding + CKKS encrypt
enc_logits = fhe_model(enc_image)         # all rotations dispatched to GPU
logits = ctx.decrypt(enc_logits)[:10]
print("Predicted:", int(np.argmax(logits)))
```

The `fhe_sdk.nn.SquareCNN` class mirrors the plaintext `SquareCNN` topology
but each operator dispatches to a GPU-backed HEonGPU primitive. No manual
weight extraction or im2col bookkeeping is needed by the caller.

### Comparison table

| Axis | TenSEAL (CPU) | Concrete-ML (CPU) | `fhe_sdk` (GPU, target) |
|------|--------------|-------------------|-------------------------|
| Conv2d support | Yes (im2col) | Quantised Conv2d | Yes (diagonal GPU) |
| Activation | CKKS poly approx | Quantised int | CKKS square |
| Latency (CNN, ~) | 30–300 s/sample | 1–60 s/sample | < 5 s/sample (target) |
| Noise model | Approximate (CKKS) | Exact (TFHE) | Approximate (CKKS) |
| GPU support | No | Partial | Yes (HEonGPU) |
| API | Low-level vectors | sklearn-like | PyTorch-like |

> **Conclusion.** TenSEAL establishes the **CPU CKKS baseline** for CNN
> inference against which `fhe_sdk` GPU performance will be reported in
> the thesis. The dominant cost is the diagonal convolution (rotation-based)
> — precisely the operation that GPU NTT acceleration addresses most directly.